In [1]:
# Create dataframes that approximate the counts in the screenshots
import pandas as pd
# 1) Total complaints by airline (for the first bar chart)
airline_totals = pd.DataFrame({
    "Airline": [
        "United", "US Airways", "American", "Southwest", "Delta", "Virgin America"
    ],
    "Count": [2400, 2200, 2000, 1200, 1000, 100]
})

# 2) Complaint categories for specific airlines (for the other bar charts)
problems = [
    "Customer Service Issue",
    "Late Flight",
    "Lost Luggage",
    "Bad Flight",
    "Cancelled Flight",
    "Flight Attendant Complaints",
    "Flight Booking Problems",
    "longlines",
    "Damaged Luggage"
]

united_counts = [700, 560, 300, 230, 190, 150, 140, 70, 30]
usair_counts  = [850, 600, 180, 100, 200, 140, 130, 70, 20]
virgin_counts = [60, 12, 5, 15, 13, 6, 18, 4, 4]

per_airline_problem = pd.DataFrame({
    "Airline": (["United"] * len(problems)) + (["US Airways"] * len(problems)) + (["Virgin America"] * len(problems)),
    "Problem": problems * 3,
    "Count": united_counts + usair_counts + virgin_counts
})

# Optional: check rough sums (not meant to match airline_totals exactly; just approximate)
rough_sums = per_airline_problem.groupby("Airline")["Count"].sum().reset_index(name="Approx_Sum_From_Problems")

# Show dataframes to the user
airline_totals, per_airline_problem, rough_sums

(          Airline  Count
 0          United   2400
 1      US Airways   2200
 2        American   2000
 3       Southwest   1200
 4           Delta   1000
 5  Virgin America    100,
            Airline                      Problem  Count
 0           United       Customer Service Issue    700
 1           United                  Late Flight    560
 2           United                 Lost Luggage    300
 3           United                   Bad Flight    230
 4           United             Cancelled Flight    190
 5           United  Flight Attendant Complaints    150
 6           United      Flight Booking Problems    140
 7           United                    longlines     70
 8           United              Damaged Luggage     30
 9       US Airways       Customer Service Issue    850
 10      US Airways                  Late Flight    600
 11      US Airways                 Lost Luggage    180
 12      US Airways                   Bad Flight    100
 13      US Airways             C

In [2]:
airline_totals

,Airline,Count
0,United,2400
1,US Airways,2200
2,American,2000
3,Southwest,1200
4,Delta,1000
5,Virgin America,100


In [4]:
import pandas as pd
import plotly.express as px

# --- Function to simplify categories ---
def simplify_categories(df, top_n=5):
    pieces = []
    for airline, g in df.groupby("Airline"):
        g_sorted = g.sort_values("Count", ascending=False)
        top = g_sorted.head(top_n)
        other_sum = g_sorted.iloc[top_n:]["Count"].sum()
        if other_sum > 0:
            top = pd.concat([top, pd.DataFrame({"Airline":[airline], "Problem":["Other"], "Count":[other_sum]})])
        pieces.append(top)
    return pd.concat(pieces, ignore_index=True)

simplified = simplify_categories(per_airline_problem, top_n=5)

# --- Faceted bar chart ---
fig = px.bar(
    simplified,
    x="Count", y="Problem",
    facet_col="Airline",
    orientation="h",
    title="Airline Complaint Categories (Top Issues + Other)",
    text="Count"
)

# Simplify look
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, height=500, width=1100)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))  # clean facet titles

fig.show()

In [5]:
import pandas as pd
import plotly.express as px

# --- Aggregate categories (top N + "Other") ---
def simplify_categories(df, top_n=5):
    pieces = []
    for airline, g in df.groupby("Airline"):
        g_sorted = g.sort_values("Count", ascending=False)
        top = g_sorted.head(top_n)
        other_sum = g_sorted.iloc[top_n:]["Count"].sum()
        if other_sum > 0:
            top = pd.concat([top, pd.DataFrame({"Airline":[airline], "Problem":["Other"], "Count":[other_sum]})])
        pieces.append(top)
    return pd.concat(pieces, ignore_index=True)

simplified = simplify_categories(per_airline_problem, top_n=5)

# --- Merge with airline totals ---
totals = airline_totals.rename(columns={"Count": "TotalComplaints"})
totals["Problem"] = "All Complaints"
totals = totals.rename(columns={"TotalComplaints": "Count"})

combined = pd.concat([simplified, totals], ignore_index=True)

# --- Single aggregated plot ---
fig = px.bar(
    combined,
    x="Airline", y="Count",
    color="Problem",
    title="Airline Complaints: Total vs. Top Categories (Simplified)",
    text="Count"
)

fig.update_traces(textposition="outside")
fig.update_layout(barmode="stack", height=600, width=800)

fig.show()

In [6]:
import pandas as pd

combined_df = pd.DataFrame({
    "Airline": [
        # --- Totals ---
        "United","US Airways","American","Southwest","Delta","Virgin America",
        # --- United categories ---
        "United","United","United","United","United","United","United","United","United",
        # --- US Airways categories ---
        "US Airways","US Airways","US Airways","US Airways","US Airways","US Airways","US Airways","US Airways","US Airways",
        # --- Virgin America categories ---
        "Virgin America","Virgin America","Virgin America","Virgin America","Virgin America",
        "Virgin America","Virgin America","Virgin America","Virgin America",
        # --- American categories (spoofed) ---
        "American","American","American","American","American","American","American","American","American",
        # --- Southwest categories (spoofed) ---
        "Southwest","Southwest","Southwest","Southwest","Southwest","Southwest","Southwest","Southwest","Southwest",
        # --- Delta categories (spoofed) ---
        "Delta","Delta","Delta","Delta","Delta","Delta","Delta","Delta","Delta"
    ],
    "Problem": [
        # --- Totals ---
        "All Complaints","All Complaints","All Complaints","All Complaints","All Complaints","All Complaints",
        # --- United categories ---
        "Customer Service Issue","Late Flight","Lost Luggage","Bad Flight","Cancelled Flight",
        "Flight Attendant Complaints","Flight Booking Problems","longlines","Damaged Luggage",
        # --- US Airways categories ---
        "Customer Service Issue","Late Flight","Lost Luggage","Bad Flight","Cancelled Flight",
        "Flight Attendant Complaints","Flight Booking Problems","longlines","Damaged Luggage",
        # --- Virgin America categories ---
        "Customer Service Issue","Late Flight","Lost Luggage","Bad Flight","Cancelled Flight",
        "Flight Attendant Complaints","Flight Booking Problems","longlines","Damaged Luggage",
        # --- American categories (spoofed) ---
        "Customer Service Issue","Late Flight","Lost Luggage","Bad Flight","Cancelled Flight",
        "Flight Attendant Complaints","Flight Booking Problems","longlines","Damaged Luggage",
        # --- Southwest categories (spoofed) ---
        "Customer Service Issue","Late Flight","Lost Luggage","Bad Flight","Cancelled Flight",
        "Flight Attendant Complaints","Flight Booking Problems","longlines","Damaged Luggage",
        # --- Delta categories (spoofed) ---
        "Customer Service Issue","Late Flight","Lost Luggage","Bad Flight","Cancelled Flight",
        "Flight Attendant Complaints","Flight Booking Problems","longlines","Damaged Luggage"
    ],
    "Count": [
        # --- Totals ---
        2400,2200,2000,1200,1000,100,
        # --- United categories ---
        700,560,300,230,190,150,140,70,30,
        # --- US Airways categories ---
        850,600,180,100,200,140,130,70,20,
        # --- Virgin America categories ---
        60,12,5,15,13,6,18,4,4,
        # --- American categories (spoofed total 2000) ---
        583,473,223,180,143,110,107,54,127,
        # --- Southwest categories (spoofed total 1200) ---
        350,284,134,108,86,66,64,32,76,
        # --- Delta categories (spoofed total 1000) ---
        291,236,111,89,71,55,54,27,66
    ],
    "Type": [
        # --- Totals ---
        "Total","Total","Total","Total","Total","Total",
        # --- Categories ---
        "Category","Category","Category","Category","Category","Category","Category","Category","Category",
        "Category","Category","Category","Category","Category","Category","Category","Category","Category",
        "Category","Category","Category","Category","Category","Category","Category","Category","Category",
        "Category","Category","Category","Category","Category","Category","Category","Category","Category",
        "Category","Category","Category","Category","Category","Category","Category","Category","Category",
        "Category","Category","Category","Category","Category","Category","Category","Category","Category"
    ]
})

combined_df

,Airline,Problem,Count,Type
0,United,All Complaints,2400,Total
1,US Airways,All Complaints,2200,Total
2,American,All Complaints,2000,Total
3,Southwest,All Complaints,1200,Total
4,Delta,All Complaints,1000,Total
5,Virgin America,All Complaints,100,Total
6,United,Customer Service Issue,700,Category
7,United,Late Flight,560,Category
8,United,Lost Luggage,300,Category
9,United,Bad Flight,230,Category


In [8]:
import plotly.express as px

# Separate totals and categories
totals_df = combined_df[combined_df["Type"] == "Total"]
cats_df   = combined_df[combined_df["Type"] == "Category"]

# --- Total complaints per airline ---
fig_totals = px.bar(
    totals_df,
    x="Airline", y="Count",
    title="Total Complaints by Airline",
    text="Count"
)
fig_totals.update_traces(textposition="outside")
fig_totals.update_layout(height=500, width=800)
fig_totals.show()

# --- Complaint categories per airline (stacked) ---
fig_cats = px.bar(
    cats_df,
    x="Airline", y="Count",
    color="Problem",
    title="Complaint Categories by Airline",
    text="Count"
)
fig_cats.update_traces(textposition="inside")
fig_cats.update_layout(barmode="stack", height=600, width=900)
fig_cats.show()

In [10]:
import pandas as pd
import plotly.express as px

# --- 1) Sort airlines by total complaints (desc) for consistent ordering ---
totals_df = combined_df[combined_df["Type"] == "Total"].copy()
cats_df   = combined_df[combined_df["Type"] == "Category"].copy()

airline_order = (
    totals_df.sort_values("Count", ascending=False)["Airline"]
    .tolist()
)

# --- 2) Combine bottom 4 categories per airline into "Other" ---
def combine_bottom_n_to_other(df, n_bottom=4):
    out = []
    for al, g in df.groupby("Airline", as_index=False):
        g_sorted = g.sort_values("Count", ascending=False)
        keep = g_sorted.iloc[:-n_bottom] if len(g_sorted) > n_bottom else g_sorted.iloc[0:0]
        bottom = g_sorted.iloc[-n_bottom:] if len(g_sorted) >= n_bottom else g_sorted
        other_sum = bottom["Count"].sum()
        if not keep.empty:
            out.append(keep)
        # Always add "Other" to represent the combined bottom categories
        out.append(pd.DataFrame({
            "Airline": [al],
            "Problem": ["Other"],
            "Count": [int(other_sum)]
        }))
    return pd.concat(out, ignore_index=True)

cats_simplified = combine_bottom_n_to_other(cats_df, n_bottom=5)

# --- 4) Stacked categories with bottom 4 combined into "Other" (same ordering) ---
fig_cats = px.bar(
    cats_simplified,
    x="Airline", y="Count",
    color="Problem",
    category_orders={"Airline": airline_order},
    title="Complaint Categories by Airline (bottom 4 combined into 'Other')",
    text="Count"
)
fig_cats.update_traces(textposition="inside")
fig_cats.update_layout(barmode="stack", height=550, width=950, legend_title_text="")
fig_cats.show()

In [13]:
import plotly.express as px

# --- 1) Sort airlines by total complaints (desc) ---
totals_df = combined_df[combined_df["Type"] == "Total"].copy()
cats_df   = combined_df[combined_df["Type"] == "Category"].copy()

airline_order = (
    totals_df.sort_values("Count", ascending=False)["Airline"]
    .tolist()
)

# --- 2) Keep only top N categories per airline (no "Other") ---
def top_n_categories(df, n_top=5):
    out = []
    for al, g in df.groupby("Airline", as_index=False):
        g_sorted = g.sort_values("Count", ascending=False)
        out.append(g_sorted.head(n_top))
    return pd.concat(out, ignore_index=True)

cats_top = top_n_categories(cats_df, n_top=4)

# --- 4) Complaint categories (top only, no 'Other') ---
fig_cats = px.bar(
    cats_top,
    x="Airline", y="Count",
    color="Problem",
    category_orders={"Airline": airline_order},
    text="Count"
)
fig_cats.update_traces(textposition="inside")
fig_cats.update_layout(barmode="stack", height=550, width=950, legend_title_text="")
fig_cats.show()

In [16]:
import plotly.express as px

# --- 1) Sort airlines by total complaints (desc) ---
totals_df = combined_df[combined_df["Type"] == "Total"].copy()
cats_df   = combined_df[combined_df["Type"] == "Category"].copy()

airline_order = (
    totals_df.sort_values("Count", ascending=False)["Airline"]
    .tolist()
)

# --- 2) Keep only top N categories per airline (no 'Other') ---
def top_n_categories(df, n_top=5):
    out = []
    for al, g in df.groupby("Airline", as_index=False):
        g_sorted = g.sort_values("Count", ascending=False)
        out.append(g_sorted.head(n_top))
    return pd.concat(out, ignore_index=True)

cats_top = top_n_categories(cats_df, n_top=4)


# --- 4) Complaint categories (top only, no axis labels) ---
fig_cats = px.bar(
    cats_top,
    x="Airline", y="Count",
    color="Problem",
    category_orders={"Airline": airline_order},
    text="Count"
)
fig_cats.update_traces(textposition="inside")
fig_cats.update_layout(
    barmode="stack", height=550, width=950,
    legend_title_text="",
    xaxis_title=None, yaxis_title=None
)
fig_cats.show()

In [18]:
import plotly.express as px

# --- 1) Sort airlines by total complaints (desc) ---
totals_df = combined_df[combined_df["Type"] == "Total"].copy()
cats_df   = combined_df[combined_df["Type"] == "Category"].copy()

airline_order = (
    totals_df.sort_values("Count", ascending=False)["Airline"]
    .tolist()
)

# --- 2) Keep only top N categories per airline (no 'Other') ---
def top_n_categories(df, n_top=5):
    out = []
    for al, g in df.groupby("Airline", as_index=False):
        g_sorted = g.sort_values("Count", ascending=False)
        out.append(g_sorted.head(n_top))
    return pd.concat(out, ignore_index=True)

cats_top = top_n_categories(cats_df, n_top=4)

# --- 4) Complaint categories (legend inside top-right, no gridlines) ---
fig_cats = px.bar(
    cats_top,
    x="Airline", y="Count",
    color="Problem",
    category_orders={"Airline": airline_order},
    text="Count"
)
fig_cats.update_traces(textposition="inside")
fig_cats.update_layout(
    barmode="stack", height=550, width=950,
    legend_title_text="",
    xaxis_title=None, yaxis_title=None,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
    legend=dict(
        x=0.98, y=0.98,
        xanchor="right", yanchor="top",
        bgcolor="rgba(255,255,255,0.6)"  # semi-transparent background
    )
)
fig_cats.show()

In [19]:
import pandas as pd
import plotly.express as px

# --- Prep: split totals & categories; get airline order (desc by total) ---
totals_df = combined_df[combined_df["Type"] == "Total"].copy()
cats_df   = combined_df[combined_df["Type"] == "Category"].copy()

airline_order = (
    totals_df.sort_values("Count", ascending=False)["Airline"].tolist()
)

# (Optional) keep only a subset of categories if you want a cleaner display:
# keep_cats = ["Customer Service Issue","Late Flight","Lost Luggage","Bad Flight","Cancelled Flight","Flight Booking Problems"]
# cats_df = cats_df[cats_df["Problem"].isin(keep_cats)]

# Common layout kwargs to simplify look
common_layout = dict(
    xaxis_title=None, yaxis_title=None,
    xaxis=dict(showgrid=False), yaxis=dict(showgrid=False),
    legend_title_text=""
)

# 1) GROUPED BAR (side-by-side per airline)
fig_grouped = px.bar(
    cats_df, x="Airline", y="Count", color="Problem",
    barmode="group",
    category_orders={"Airline": airline_order},
    title="Complaints by Airline • Grouped by Category",
    text="Count"
)
fig_grouped.update_traces(textposition="outside", cliponaxis=False)
fig_grouped.update_layout(height=550, width=1000, **common_layout)
fig_grouped.show()

# 2) 100% STACKED BAR (proportions per airline)
cats_pct = cats_df.merge(
    totals_df[["Airline","Count"]].rename(columns={"Count":"Total"}),
    on="Airline", how="left"
).assign(Percent=lambda d: d["Count"]/d["Total"])

fig_100 = px.bar(
    cats_pct, x="Airline", y="Percent", color="Problem",
    barmode="stack",
    category_orders={"Airline": airline_order},
    text=cats_pct["Percent"].map(lambda p: f"{p:.0%}"),
    title="Complaints by Airline • 100% Stacked (Proportions)"
)
fig_100.update_traces(textposition="inside")
fig_100.update_layout(height=550, width=1000, yaxis_tickformat=".0%", **common_layout)
fig_100.show()

# 3) HEATMAP (easy cross-airline comparison by category)
pivot = cats_df.pivot_table(index="Problem", columns="Airline", values="Count", aggfunc="sum").fillna(0)
# Reorder columns (airlines) by total complaints
pivot = pivot[airline_order]

fig_heat = px.imshow(
    pivot,
    aspect="auto",
    color_continuous_scale="Blues",
    title="Heatmap • Complaint Counts by Category and Airline",
    labels=dict(color="Count")
)
fig_heat.update_layout(height=600, width=950)
fig_heat.show()

# 4) TREEMAP (share across industry → airline → category)
fig_tree = px.treemap(
    cats_df, path=["Airline", "Problem"], values="Count",
    color="Airline",  # just to separate visually
    title="Treemap • Complaint Composition by Airline and Category"
)
fig_tree.update_layout(height=600, width=950)
fig_tree.show()

# 5) DOT / SCATTER (one dot per airline/category)
fig_dot = px.scatter(
    cats_df,
    x="Airline", y="Count", color="Problem",
    category_orders={"Airline": airline_order},
    title="Dot Plot • Complaint Counts by Airline & Category",
    hover_data=["Problem","Airline","Count"]
)
fig_dot.update_layout(height=500, width=950, **common_layout)
fig_dot.show()

In [20]:
import pandas as pd
import plotly.express as px

# --- Find top 3 complaint categories overall ---
top3 = (
    cats_df.groupby("Problem")["Count"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)

# --- Filter dataset to top 3 categories ---
cats_top3 = cats_df[cats_df["Problem"].isin(top3)]

# --- Pivot for heatmap ---
pivot_top3 = cats_top3.pivot_table(
    index="Problem", columns="Airline", values="Count", aggfunc="sum"
).fillna(0)

# Reorder airline columns by total complaints
pivot_top3 = pivot_top3[airline_order]

# --- Plot heatmap ---
fig_heat_top3 = px.imshow(
    pivot_top3,
    aspect="auto",
    color_continuous_scale="Blues",
    title="Heatmap • Top 3 Complaint Categories by Airline",
    labels=dict(color="Count")
)
fig_heat_top3.update_layout(height=400, width=900)
fig_heat_top3.show()

In [25]:
import plotly.express as px

# --- Find top 3 complaint categories overall ---
top3 = (
    cats_df.groupby("Problem")["Count"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)

# --- Filter dataset to top 3 categories ---
cats_top3 = cats_df[cats_df["Problem"].isin(top3)]

# --- Pivot for heatmap ---
pivot_top3 = cats_top3.pivot_table(
    index="Problem", columns="Airline", values="Count", aggfunc="sum"
).fillna(0)

# Reorder airline columns by total complaints
pivot_top3 = pivot_top3[airline_order]

# --- Plot heatmap with red scale ---
fig_heat_top3 = px.imshow(
    pivot_top3,
    aspect="auto",
    color_continuous_scale="Reds",  # high = red
    labels=dict(color="Count")
)
fig_heat_top3.update_layout(height=400, width=900)
fig_heat_top3.show()

In [26]:
import plotly.express as px

# --- Find top 3 complaint categories overall ---
top3 = (
    cats_df.groupby("Problem")["Count"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)

# --- Filter dataset to top 3 categories ---
cats_top3 = cats_df[cats_df["Problem"].isin(top3)]

# --- Pivot for heatmap ---
pivot_top3 = cats_top3.pivot_table(
    index="Problem", columns="Airline", values="Count", aggfunc="sum"
).fillna(0)

# Reorder airline columns by total complaints
pivot_top3 = pivot_top3[airline_order]

# --- Plot heatmap (red scale, no titles) ---
fig_heat_top3 = px.imshow(
    pivot_top3,
    aspect="auto",
    color_continuous_scale="Reds",
    labels=dict(color="Count")
)

# Remove titles and axis labels
fig_heat_top3.update_layout(
    height=400, width=900,
    title=None,
    xaxis_title=None, yaxis_title=None
)

fig_heat_top3.show()